# WCCI Action Distribution and Checkpoint-Style Plots

This notebook mirrors the A0 full-test/checkpoint action-distribution notebook, but targets the downloaded WCCI groups. It uses W&B history action metrics by default and automatically picks up WCCI full-test `action_summary.json` files if they are added later.

In [69]:
from pathlib import Path
import json
import re
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


def find_task_dir(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "main.py").is_file() and candidate.name == "Topology_Task":
            return candidate
        task_dir = candidate / "Topology_Task"
        if (task_dir / "main.py").is_file():
            return task_dir
    raise RuntimeError("Could not locate Topology_Task/main.py")


TASK_DIR = find_task_dir()
HELPER_DIR = TASK_DIR / "analysis" / "metrics" / "helpers"
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

from run_data import scan_run_data

RUN_DATA_ROOT = TASK_DIR / "outputs" / "run_data"
FIG_DIR = TASK_DIR / "outputs" / "wcci_metric_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

WCCI_GROUPS = [
    "wcci_aib_2",
    "wcci_sparse16_2",
    "wcci_hvg_2",
    "wcci_baseline_2",
    "wcci_gnn",
]
SEEDS = (0, 1, 2)
AGENTS = ["agent_0", "agent_1", "agent_2", "agent_3"]
SAVE_FIGURES = True
SHOW_FIGURES = True

print("Task dir:", TASK_DIR)
print("Run data root:", RUN_DATA_ROOT)
print("WCCI groups:", WCCI_GROUPS)


Task dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task
Run data root: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data
WCCI groups: ['wcci_aib_2', 'wcci_sparse16_2', 'wcci_hvg_2', 'wcci_baseline_2', 'wcci_gnn']


## Load Downloaded WCCI Histories

In [70]:
def _path_or_none(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    path = Path(value).expanduser()
    return path if path.exists() else None


def read_history(row):
    parquet_path = _path_or_none(row.get("history_parquet"))
    csv_path = _path_or_none(row.get("history_csv"))
    if parquet_path is not None:
        history = pd.read_parquet(parquet_path)
    elif csv_path is not None:
        history = pd.read_csv(csv_path)
    else:
        raise FileNotFoundError(f"No local history file for {row.get('run_name')}")
    history = history.copy()
    history["run_name"] = row["run_name"]
    history["run_id"] = row["run_id"]
    history["download_group"] = row["download_group"]
    history["run_state"] = row.get("state")
    return history


run_index_all = scan_run_data(RUN_DATA_ROOT, include_legacy=False)
run_index = run_index_all[run_index_all["download_group"].isin(WCCI_GROUPS)].copy()
if run_index.empty:
    raise RuntimeError(f"No WCCI runs found under {RUN_DATA_ROOT}. Check that the downloaded groups exist.")

histories = []
for row in run_index.to_dict("records"):
    try:
        histories.append(read_history(row))
    except Exception as exc:
        print(f"Skipped {row.get('run_name')}: {exc}")

history_df = pd.concat(histories, ignore_index=True, sort=False) if histories else pd.DataFrame()
if history_df.empty:
    raise RuntimeError("No WCCI histories could be loaded.")

coverage = (
    run_index.assign(
        family=lambda df: df["run_name"].map(lambda name: re.sub(r"_s\\d+$", "", str(name))),
        seed=lambda df: df["run_name"].map(lambda name: int(re.search(r"_s(\\d+)$", str(name)).group(1)) if re.search(r"_s(\\d+)$", str(name)) else np.nan),
    )
    .groupby(["download_group", "family", "state"], dropna=False, as_index=False)
    .agg(
        runs=("run_name", "count"),
        seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
        rows_min=("rows", "min"),
        rows_max=("rows", "max"),
    )
    .sort_values(["download_group", "family", "state"])
)
print(f"Loaded {len(run_index)} runs and {len(history_df):,} history rows.")
display(coverage)


Loaded 36 runs and 51,490 history rows.


,download_group,family,state,runs,seeds,rows_min,rows_max
0,wcci_aib_2,wcci_aib_00_flat_local_t020_72x576_lr20m_f010_s0,finished,1,[],1446,1446
1,wcci_aib_2,wcci_aib_00_flat_local_t020_72x576_lr20m_f010_s1,finished,1,[],1446,1446
2,wcci_aib_2,wcci_aib_00_flat_local_t020_72x576_lr20m_f010_s2,finished,1,[],1446,1446
3,wcci_aib_2,wcci_aib_01_flat_local_t010_72x576_lr20m_f010_s0,crashed,1,[],1355,1355
4,wcci_aib_2,wcci_aib_01_flat_local_t010_72x576_lr20m_f010_s1,crashed,1,[],1373,1373
5,wcci_aib_2,wcci_aib_01_flat_local_t010_72x576_lr20m_f010_s2,crashed,1,[],1383,1383
6,wcci_baseline_2,wcci_baseline_gnn_features_72x576_s0,finished,1,[],1446,1446
7,wcci_baseline_2,wcci_baseline_gnn_features_72x576_s1,finished,1,[],1446,1446
8,wcci_baseline_2,wcci_baseline_gnn_features_72x576_s2,finished,1,[],1446,1446
9,wcci_baseline_2,wcci_mlp_baseline_512x512x512_72x576_lr20m_f01...,finished,1,[],1446,1446


## Labels and Run Helpers

In [71]:
def safe_name(text):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(text)).strip("._-") or "plot"


def save_figure(fig, name):
    if fig is None or not SAVE_FIGURES:
        return None
    path = FIG_DIR / f"{safe_name(name)}.html"
    fig.write_html(path, include_plotlyjs="cdn")
    print("Saved:", path)
    return path


def seed_from_run(run_name):
    match = re.search(r"_s(\d+)$", str(run_name))
    return int(match.group(1)) if match else np.nan


def family_from_run(run_name):
    return re.sub(r"_s\d+$", "", str(run_name))


WCCI_LABELS = {
    "wcci_mlp_baseline_72x576": "MLP baseline 72x576",
    "wcci_mlp_baseline_72x576_lr20m_f010": "MLP baseline 72x576 lr20M f0.10",
    "wcci_mlp_baseline_512x512x512_72x576_lr20m_f010": "MLP baseline 512x3 lr20M f0.10",
    "wcci_baseline_gnn_features_72x576": "MLP baseline GNN features",
    "wcci_gine_light_nonshared_72x576": "GINE light non-shared",
    "wcci_hvg_01_eval_rho090_72x576_lr20m_f010": "global rho heuristic 0.90",
    "wcci_hvg_04_eval_local_rho090_72x576_lr20m_f010": "local rho heuristic 0.90",
    "wcci_hvg_02_gate_final_map_72x576_lr20m_f010": "gate final-action MAP",
    "wcci_hvg_03_gate_hierarchical_72x576_lr20m_f010": "gate hierarchical greedy",
    "wcci_sparse16_flat_p010_72x576_lr20m_f010": "Sparse16 flat p0.010",
    "wcci_aib_00_flat_local_t020_72x576_lr20m_f010": "AIB flat local t0.20 lr20m f0.10",
    "wcci_aib_01_flat_local_t010_72x576_lr20m_f010": "AIB flat local t0.10 lr20m f0.10",
}

WCCI_COLOR = {
    "MLP baseline 72x576": "#4e79a7",
    "MLP baseline 72x576 lr20M f0.10": "#1f77b4",
    "MLP baseline 512x3 lr20M f0.10": "#17becf",
    "MLP baseline GNN features": "#7f7f7f",
    "GINE light non-shared": "#003f5c",
    "global rho heuristic 0.90": "#ff7f0e",
    "local rho heuristic 0.90": "#2ca02c",
    "gate final-action MAP": "#9467bd",
    "gate hierarchical greedy": "#d62728",
    "Sparse16 flat p0.010": "#8c564b",
    "AIB flat local t0.20 lr20m f0.10": "#bcbd22",
    "AIB flat local t0.10 lr20m f0.10": "#e377c2",
}


def pretty_label(family):
    return WCCI_LABELS.get(str(family), str(family).replace("wcci_", "").replace("_", " "))


def seeded(prefix, seeds=SEEDS):
    return [f"{prefix}_s{seed}" for seed in seeds]


def select_plot_runs(run_groups, labels=None, run_names=None):
    """Select curve labels and, optionally, individual seed run names for one plot."""
    labels = list(run_groups) if labels is None else list(labels)
    run_name_filter = None if run_names is None else set(run_names)
    selected = {}
    for label in labels:
        runs = list(run_groups[label])
        if run_name_filter is not None:
            runs = [run for run in runs if run in run_name_filter]
        if runs:
            selected[label] = runs
    return selected


def available_run_names(history=None):
    data = history_df if history is None else history
    return set(data["run_name"].dropna().astype(str).unique())


def report_missing_runs(run_groups, history=None):
    available = available_run_names(history)
    rows = []
    for label, runs in run_groups.items():
        missing = [run for run in runs if run not in available]
        rows.append({"curve": label, "expected": len(runs), "available": len(runs) - len(missing), "missing": missing})
    coverage = pd.DataFrame(rows)
    display(coverage)
    return coverage


def infer_family_table():
    rows = []
    for run_name in sorted(history_df["run_name"].dropna().astype(str).unique()):
        rows.append({
            "download_group": history_df.loc[history_df["run_name"].eq(run_name), "download_group"].iloc[0],
            "run_name": run_name,
            "family": family_from_run(run_name),
            "label": pretty_label(family_from_run(run_name)),
            "seed": seed_from_run(run_name),
        })
    return pd.DataFrame(rows)


family_table = infer_family_table()
display(family_table.sort_values(["download_group", "family", "seed"]))


,download_group,run_name,family,label,seed
0,wcci_aib_2,wcci_aib_00_flat_local_t020_72x576_lr20m_f010_s0,wcci_aib_00_flat_local_t020_72x576_lr20m_f010,AIB flat local t0.20 lr20m f0.10,0
1,wcci_aib_2,wcci_aib_00_flat_local_t020_72x576_lr20m_f010_s1,wcci_aib_00_flat_local_t020_72x576_lr20m_f010,AIB flat local t0.20 lr20m f0.10,1
2,wcci_aib_2,wcci_aib_00_flat_local_t020_72x576_lr20m_f010_s2,wcci_aib_00_flat_local_t020_72x576_lr20m_f010,AIB flat local t0.20 lr20m f0.10,2
3,wcci_aib_2,wcci_aib_01_flat_local_t010_72x576_lr20m_f010_s0,wcci_aib_01_flat_local_t010_72x576_lr20m_f010,AIB flat local t0.10 lr20m f0.10,0
4,wcci_aib_2,wcci_aib_01_flat_local_t010_72x576_lr20m_f010_s1,wcci_aib_01_flat_local_t010_72x576_lr20m_f010,AIB flat local t0.10 lr20m f0.10,1
5,wcci_aib_2,wcci_aib_01_flat_local_t010_72x576_lr20m_f010_s2,wcci_aib_01_flat_local_t010_72x576_lr20m_f010,AIB flat local t0.10 lr20m f0.10,2
6,wcci_baseline_2,wcci_baseline_gnn_features_72x576_s0,wcci_baseline_gnn_features_72x576,MLP baseline GNN features,0
7,wcci_baseline_2,wcci_baseline_gnn_features_72x576_s1,wcci_baseline_gnn_features_72x576,MLP baseline GNN features,1
8,wcci_baseline_2,wcci_baseline_gnn_features_72x576_s2,wcci_baseline_gnn_features_72x576,MLP baseline GNN features,2
24,wcci_baseline_2,wcci_mlp_baseline_512x512x512_72x576_lr20m_f01...,wcci_mlp_baseline_512x512x512_72x576_lr20m_f010,MLP baseline 512x3 lr20M f0.10,0


## Action Distribution Helpers

In [72]:
ACTION0_METRICS = {
    "test": ["test/explain/frac_action_0_agent_{agent}"],
    "train_eval": ["train_eval/explain/frac_action_0_agent_{agent}"],
    "train": ["train/frac_action_0_agent_{agent}"],
}
SURVIVAL_METRICS_FOR_TRADEOFF = {
    "test": ["test/episodic_survival", "test/charts/episodic_survival"],
    "train_eval": ["train_eval/episodic_survival", "train_eval/charts/episodic_survival"],
}
NONIDLE_COUNT_METRICS = [f"train/non_idle_agents_count_{idx}_frac" for idx in range(5)]


def _agent_metric_id(agent):
    text = str(agent)
    return text.rsplit("_", 1)[-1] if text.startswith("agent_") else text


def existing_metric(candidates, *, agent=None, data=None):
    data = history_df if data is None else data
    for template in candidates:
        metric = template.format(agent=_agent_metric_id(agent)) if agent is not None else template
        if metric in data.columns:
            return metric
    return None


def value_scale(values, *, percent=True):
    numeric = pd.to_numeric(values, errors="coerce")
    max_value = numeric.max(skipna=True)
    if pd.isna(max_value):
        return 100.0 if percent else 1.0
    return 100.0 if percent and max_value <= 1.5 else 1.0


def latest_non_null(run_data, metric):
    if metric not in run_data.columns:
        return None
    values = run_data[["_step", metric]].dropna(subset=[metric]).sort_values("_step")
    if values.empty:
        return None
    row = values.iloc[-1]
    return float(row[metric]), int(row["_step"])


def collect_latest_action0(*, source="test", history=None):
    data = history_df if history is None else history
    rows = []
    for run_name, run_data in data.groupby("run_name", sort=False):
        family = family_from_run(run_name)
        seed = seed_from_run(run_name)
        download_group = run_data["download_group"].iloc[0] if "download_group" in run_data.columns else None
        for agent in AGENTS:
            metric = existing_metric(ACTION0_METRICS[source], agent=agent, data=run_data)
            if metric is None:
                continue
            latest = latest_non_null(run_data, metric)
            if latest is None:
                continue
            value, step = latest
            rows.append({
                "run_name": run_name,
                "family": family,
                "condition_label": pretty_label(family),
                "seed": seed,
                "download_group": download_group,
                "agent": agent,
                "metric_source": source,
                "metric": metric,
                "action0_fraction": value,
                "selected_metric_step": step,
            })
    return pd.DataFrame(rows)


def summarize_action0(action_rows):
    if action_rows.empty:
        return pd.DataFrame()
    action_rows = action_rows.copy()
    if "selected_metric_step" not in action_rows.columns:
        action_rows["selected_metric_step"] = action_rows["global_step"] if "global_step" in action_rows.columns else np.nan
    summary = (
        action_rows.groupby(["condition_label", "family", "agent"], as_index=False, observed=True)
        .agg(
            mean_action0_fraction=("action0_fraction", "mean"),
            std_action0_fraction=("action0_fraction", "std"),
            min_action0_fraction=("action0_fraction", "min"),
            max_action0_fraction=("action0_fraction", "max"),
            n_seeds=("seed", "nunique"),
            seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
            selected_metric_steps=("selected_metric_step", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
        )
    )
    summary["std_action0_fraction"] = summary["std_action0_fraction"].fillna(0.0)
    return summary


def collect_latest_survival(*, split="test", history=None):
    data = history_df if history is None else history
    candidates = SURVIVAL_METRICS_FOR_TRADEOFF[split]
    rows = []
    for run_name, run_data in data.groupby("run_name", sort=False):
        metric = existing_metric(candidates, data=run_data)
        if metric is None:
            continue
        latest = latest_non_null(run_data, metric)
        if latest is None:
            continue
        value, step = latest
        scale = value_scale(run_data[metric], percent=True)
        family = family_from_run(run_name)
        download_group = run_data["download_group"].iloc[0] if "download_group" in run_data.columns else None
        rows.append({
            "run_name": run_name,
            "family": family,
            "download_group": download_group,
            "condition_label": pretty_label(family),
            "seed": seed_from_run(run_name),
            "survival_percent": value * scale,
            "survival_metric": metric,
            "selected_survival_step": step,
        })
    return pd.DataFrame(rows)


def _families_from_run_groups(run_groups):
    families = []
    for runs in run_groups.values():
        for run in runs:
            family = family_from_run(run)
            if family not in families:
                families.append(family)
    return families


def _order_labels(labels):
    return sorted(labels, key=lambda label: (list(WCCI_COLOR).index(label) if label in WCCI_COLOR else 999, label))


def plot_action0_by_agent(action_rows, run_groups, *, title, save_name=None):
    families = set(_families_from_run_groups(run_groups))
    rows = action_rows[action_rows["family"].isin(families)].copy()
    if rows.empty:
        print(f"No action-0 rows for {title}")
        return None, rows, pd.DataFrame()
    if "selected_metric_step" not in rows.columns:
        rows["selected_metric_step"] = rows["global_step"] if "global_step" in rows.columns else np.nan
    if "metric" not in rows.columns:
        rows["metric"] = ""
    if "download_group" not in rows.columns:
        rows["download_group"] = ""
    summary = summarize_action0(rows)
    label_order = [label for label in run_groups.keys() if label in set(summary["condition_label"])]
    label_order += [label for label in _order_labels(summary["condition_label"].unique()) if label not in label_order]
    agent_order = [agent for agent in AGENTS if agent in set(summary["agent"])]

    x_index = {label: idx for idx, label in enumerate(label_order)}
    agent_width = 0.72 / max(len(agent_order), 1)
    agent_offsets = {
        agent: (idx - (len(agent_order) - 1) / 2.0) * agent_width
        for idx, agent in enumerate(agent_order)
    }
    bar_width = agent_width * 0.82
    agent_pattern_cycle = ["", "/", "x", "."]
    agent_patterns = {agent: agent_pattern_cycle[idx % len(agent_pattern_cycle)] for idx, agent in enumerate(agent_order)}
    seed_marker_symbols = {0: "circle", 1: "diamond", 2: "square", 3: "cross", 4: "x"}
    palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24
    colors = {label: WCCI_COLOR.get(label, palette[idx % len(palette)]) for idx, label in enumerate(label_order)}

    fig = go.Figure()
    for label in label_order:
        label_summary = summary[summary["condition_label"].astype(str).eq(str(label))].set_index("agent")
        x_values = []
        y_values = []
        text_values = []
        customdata = []
        pattern_shapes = []
        for agent in agent_order:
            x_values.append(x_index[label] + agent_offsets[agent])
            pattern_shapes.append(agent_patterns[agent])
            if agent in label_summary.index:
                row = label_summary.loc[agent]
                y_value = float(row["mean_action0_fraction"])
                y_values.append(y_value)
                text_values.append(f"{y_value:.3f}")
                customdata.append([
                    agent,
                    row["family"],
                    int(row["n_seeds"]),
                    str(row["seeds"]),
                    str(row["selected_metric_steps"]),
                    float(row["std_action0_fraction"]),
                    float(row["min_action0_fraction"]),
                    float(row["max_action0_fraction"]),
                ])
            else:
                y_values.append(None)
                text_values.append("")
                customdata.append([agent, "", 0, "[]", "[]", np.nan, np.nan, np.nan])
        fig.add_trace(go.Bar(
            x=x_values,
            y=y_values,
            width=bar_width,
            name=label,
            legendgroup=label,
            marker={
                "color": colors[label],
                "line": {"color": "rgba(0,0,0,0.35)", "width": 0.8},
                "pattern": {"shape": pattern_shapes, "fgcolor": "rgba(255,255,255,0.75)", "size": 8},
            },
            text=text_values,
            textposition="outside",
            cliponaxis=False,
            customdata=np.array(customdata, dtype=object),
            hovertemplate=(
                f"run type={label}<br>"
                "agent=%{customdata[0]}<br>"
                "mean action-0 fraction=%{y:.4f}<br>"
                "std action-0=%{customdata[5]:.4f}<br>"
                "min action-0=%{customdata[6]:.4f}<br>"
                "max action-0=%{customdata[7]:.4f}<br>"
                "n_seeds=%{customdata[2]}<br>"
                "seeds=%{customdata[3]}<br>"
                "selected_metric_steps=%{customdata[4]}<extra></extra>"
            ),
        ))

    for (label, agent), group in rows.groupby(["condition_label", "agent"], observed=True, sort=False):
        label = str(label)
        agent = str(agent)
        if label not in x_index or agent not in agent_offsets:
            continue
        group = group.sort_values("seed").copy()
        if group.empty:
            continue
        jitter = np.linspace(-bar_width * 0.22, bar_width * 0.22, len(group)) if len(group) > 1 else np.array([0.0])
        x_center = x_index[label] + agent_offsets[agent]
        fig.add_trace(go.Scatter(
            x=x_center + jitter,
            y=group["action0_fraction"],
            mode="markers",
            name=f"{label} {agent} seeds",
            legendgroup=label,
            showlegend=False,
            marker={
                "color": colors[label],
                "size": 9,
                "symbol": [seed_marker_symbols.get(int(seed), "circle") if pd.notna(seed) else "circle" for seed in group["seed"]],
                "line": {"color": "white", "width": 1.1},
                "opacity": 0.95,
            },
            customdata=group[["run_name", "seed", "selected_metric_step", "metric", "download_group"]].to_numpy(dtype=object),
            hovertemplate=(
                "run=%{customdata[0]}<br>"
                "seed=%{customdata[1]}<br>"
                f"agent={agent}<br>"
                f"run type={label}<br>"
                "action-0 fraction=%{y:.4f}<br>"
                "selected_metric_step=%{customdata[2]}<br>"
                "metric=%{customdata[3]}<br>"
                "download_group=%{customdata[4]}<extra></extra>"
            ),
        ))
    fig.update_layout(
        title=title,
        template="plotly_white",
        width=1450,
        height=650,
        bargap=0.20,
        legend={"title": {"text": "run variant"}, "orientation": "v", "yanchor": "top", "y": 1, "xanchor": "left", "x": 1.01},
        margin={"l": 80, "r": 250, "t": 90, "b": 145},
    )
    fig.update_xaxes(tickmode="array", tickvals=list(range(len(label_order))), ticktext=label_order, title_text="run")
    fig.update_yaxes(range=[0, 1.02], title_text="action 0 fraction")
    fig.add_annotation(
        text="Bars show means over seeds. Dots show individual seed values. No uncertainty bars are drawn. Bar patterns identify agents.",
        xref="paper",
        yref="paper",
        x=0,
        y=-0.22,
        showarrow=False,
        align="left",
        font={"size": 12, "color": "#555"},
    )
    save_figure(fig, save_name or title)
    if SHOW_FIGURES:
        fig.show()
    return fig, rows, summary


def collect_nonidle_count_distribution(history=None):
    data = history_df if history is None else history
    rows = []
    for run_name, run_data in data.groupby("run_name", sort=False):
        family = family_from_run(run_name)
        seed = seed_from_run(run_name)
        download_group = run_data["download_group"].iloc[0] if "download_group" in run_data.columns else None
        for metric in NONIDLE_COUNT_METRICS:
            if metric not in run_data.columns:
                continue
            latest = latest_non_null(run_data, metric)
            if latest is None:
                continue
            value, step = latest
            count = int(re.search(r"count_(\d+)_frac", metric).group(1))
            rows.append({
                "run_name": run_name,
                "family": family,
                "condition_label": pretty_label(family),
                "seed": seed,
                "download_group": download_group,
                "nonidle_agent_count": count,
                "fraction": value,
                "selected_metric_step": step,
            })
    return pd.DataFrame(rows)


def plot_nonidle_distribution(nonidle_rows, run_groups, *, title, save_name=None):
    families = set(_families_from_run_groups(run_groups))
    rows = nonidle_rows[nonidle_rows["family"].isin(families)].copy()
    if rows.empty:
        print(f"No non-idle count rows for {title}")
        return None, rows, pd.DataFrame()
    if "selected_metric_step" not in rows.columns:
        rows["selected_metric_step"] = np.nan
    if "download_group" not in rows.columns:
        rows["download_group"] = ""
    summary = (
        rows.groupby(["condition_label", "family", "nonidle_agent_count"], as_index=False, observed=True)
        .agg(
            mean_fraction=("fraction", "mean"),
            std_fraction=("fraction", "std"),
            n_seeds=("seed", "nunique"),
            seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
            selected_metric_steps=("selected_metric_step", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
        )
    )
    summary["std_fraction"] = summary["std_fraction"].fillna(0.0)
    label_order = [label for label in run_groups.keys() if label in set(summary["condition_label"])]
    label_order += [label for label in _order_labels(summary["condition_label"].unique()) if label not in label_order]
    count_order = sorted(pd.Series(summary["nonidle_agent_count"]).dropna().astype(int).unique().tolist())
    x_index = {label: idx for idx, label in enumerate(label_order)}
    count_width = 0.72 / max(len(count_order), 1)
    count_offsets = {
        count: (idx - (len(count_order) - 1) / 2.0) * count_width
        for idx, count in enumerate(count_order)
    }
    bar_width = count_width * 0.82
    palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24
    count_colors = {count: palette[idx % len(palette)] for idx, count in enumerate(count_order)}
    seed_marker_symbols = {0: "circle", 1: "diamond", 2: "square", 3: "cross", 4: "x"}

    fig = go.Figure()
    for count in count_order:
        count_summary = summary[summary["nonidle_agent_count"].astype(int).eq(count)].set_index("condition_label")
        x_values = []
        y_values = []
        text_values = []
        customdata = []
        for label in label_order:
            x_values.append(x_index[label] + count_offsets[count])
            if label in count_summary.index:
                row = count_summary.loc[label]
                y_value = float(row["mean_fraction"])
                y_values.append(y_value)
                text_values.append(f"{y_value:.3f}")
                customdata.append([
                    label,
                    count,
                    row["family"],
                    int(row["n_seeds"]),
                    str(row["seeds"]),
                    str(row["selected_metric_steps"]),
                    float(row["std_fraction"]),
                ])
            else:
                y_values.append(None)
                text_values.append("")
                customdata.append([label, count, "", 0, "[]", "[]", np.nan])
        fig.add_trace(go.Bar(
            x=x_values,
            y=y_values,
            width=bar_width,
            name=str(count),
            legendgroup=f"count {count}",
            marker={"color": count_colors[count], "line": {"color": "rgba(0,0,0,0.35)", "width": 0.8}},
            text=text_values,
            textposition="outside",
            cliponaxis=False,
            customdata=np.array(customdata, dtype=object),
            hovertemplate=(
                "run=%{customdata[0]}<br>"
                "non-idle agents=%{customdata[1]}<br>"
                "mean fraction=%{y:.4f}<br>"
                "std fraction=%{customdata[6]:.4f}<br>"
                "n_seeds=%{customdata[3]}<br>"
                "seeds=%{customdata[4]}<br>"
                "selected_metric_steps=%{customdata[5]}<extra></extra>"
            ),
        ))

    for (label, count), group in rows.groupby(["condition_label", "nonidle_agent_count"], observed=True, sort=False):
        label = str(label)
        count = int(count)
        if label not in x_index or count not in count_offsets:
            continue
        group = group.sort_values("seed").copy()
        jitter = np.linspace(-bar_width * 0.22, bar_width * 0.22, len(group)) if len(group) > 1 else np.array([0.0])
        x_center = x_index[label] + count_offsets[count]
        fig.add_trace(go.Scatter(
            x=x_center + jitter,
            y=group["fraction"],
            mode="markers",
            name=f"{label} count {count} seeds",
            legendgroup=f"count {count}",
            showlegend=False,
            marker={
                "color": count_colors[count],
                "size": 9,
                "symbol": [seed_marker_symbols.get(int(seed), "circle") if pd.notna(seed) else "circle" for seed in group["seed"]],
                "line": {"color": "white", "width": 1.1},
                "opacity": 0.95,
            },
            customdata=group[["run_name", "seed", "selected_metric_step", "download_group"]].to_numpy(dtype=object),
            hovertemplate=(
                "run=%{customdata[0]}<br>"
                "seed=%{customdata[1]}<br>"
                f"non-idle agents={count}<br>"
                f"run type={label}<br>"
                "fraction=%{y:.4f}<br>"
                "selected_metric_step=%{customdata[2]}<br>"
                "download_group=%{customdata[3]}<extra></extra>"
            ),
        ))
    fig.update_layout(
        title=title,
        template="plotly_white",
        width=1450,
        height=620,
        legend={"title": {"text": "# non-idle agents"}, "orientation": "v", "yanchor": "top", "y": 1, "xanchor": "left", "x": 1.01},
        margin={"l": 80, "r": 220, "t": 90, "b": 145},
    )
    fig.update_xaxes(tickmode="array", tickvals=list(range(len(label_order))), ticktext=label_order, title_text="run")
    fig.update_yaxes(range=[0, 1.02], title_text="fraction")
    fig.add_annotation(
        text="Bars show means over seeds. Dots show individual seed values. No uncertainty bars are drawn.",
        xref="paper",
        yref="paper",
        x=0,
        y=-0.22,
        showarrow=False,
        align="left",
        font={"size": 12, "color": "#555"},
    )
    save_figure(fig, save_name or title)
    if SHOW_FIGURES:
        fig.show()
    return fig, rows, summary


def plot_action0_survival_tradeoff(action_rows, survival_rows, run_groups, *, title, save_name=None):
    families = set(_families_from_run_groups(run_groups))
    action = action_rows[action_rows["family"].isin(families)].copy()
    survival = survival_rows[survival_rows["family"].isin(families)].copy()
    if action.empty or survival.empty:
        print(f"Missing action or survival rows for {title}")
        return None, pd.DataFrame()
    action_all = (
        action.groupby(["run_name", "family", "condition_label", "seed"], as_index=False, observed=True)
        .agg(action0_all_agents=("action0_fraction", "mean"))
    )
    merged = action_all.merge(survival, on=["run_name", "family", "condition_label", "seed"], how="inner")
    if merged.empty:
        print(f"No merged action/survival rows for {title}")
        return None, merged
    mean_rows = (
        merged.groupby(["family", "condition_label"], as_index=False, observed=True)
        .agg(
            mean_action0_all_agents=("action0_all_agents", "mean"),
            std_action0_all_agents=("action0_all_agents", "std"),
            mean_survival_percent=("survival_percent", "mean"),
            std_survival_percent=("survival_percent", "std"),
            n_seeds=("seed", "nunique"),
            seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
        )
    )
    label_order = [label for label in run_groups.keys() if label in set(mean_rows["condition_label"])]
    label_order += [label for label in _order_labels(mean_rows["condition_label"].unique()) if label not in label_order]
    palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24
    colors = {label: WCCI_COLOR.get(label, palette[idx % len(palette)]) for idx, label in enumerate(label_order)}
    seed_marker_symbols = {0: "circle", 1: "diamond", 2: "square", 3: "cross", 4: "x"}

    fig = go.Figure()
    for label in label_order:
        seed_points = merged[merged["condition_label"].astype(str).eq(str(label))].sort_values("seed")
        if not seed_points.empty:
            fig.add_trace(go.Scatter(
                x=seed_points["action0_all_agents"],
                y=seed_points["survival_percent"],
                mode="markers",
                name=f"{label} seeds",
                legendgroup=label,
                showlegend=False,
                marker={
                    "color": colors[label],
                    "size": 7,
                    "symbol": [seed_marker_symbols.get(int(seed), "circle") if pd.notna(seed) else "circle" for seed in seed_points["seed"]],
                    "line": {"color": "white", "width": 1.0},
                    "opacity": 0.55,
                },
                customdata=seed_points[["run_name", "seed", "survival_metric", "selected_survival_step"]].to_numpy(dtype=object),
                hovertemplate=(
                    "run=%{customdata[0]}<br>"
                    "seed=%{customdata[1]}<br>"
                    f"run type={label}<br>"
                    "action-0=%{x:.4f}<br>"
                    "survival=%{y:.2f}%<br>"
                    "survival_metric=%{customdata[2]}<br>"
                    "selected_survival_step=%{customdata[3]}<extra></extra>"
                ),
            ))

        label_mean = mean_rows[mean_rows["condition_label"].astype(str).eq(str(label))]
        if label_mean.empty:
            continue
        row = label_mean.iloc[0]
        fig.add_trace(go.Scatter(
            x=[row["mean_action0_all_agents"]],
            y=[row["mean_survival_percent"]],
            mode="markers+text",
            name=label,
            legendgroup=label,
            showlegend=True,
            text=[label],
            textposition="top center",
            marker={"color": colors[label], "size": 12, "symbol": "diamond", "line": {"color": "black", "width": 1.1}},
            customdata=np.array([[
                row["family"],
                int(row["n_seeds"]),
                str(row["seeds"]),
                float(row["std_action0_all_agents"]),
                float(row["std_survival_percent"]),
            ]], dtype=object),
            hovertemplate=(
                f"<b>{label}</b><br>"
                "mean action-0=%{x:.4f}<br>"
                "std action-0=%{customdata[3]:.4f}<br>"
                "mean survival=%{y:.2f}%<br>"
                "std survival=%{customdata[4]:.2f}%<br>"
                "n_seeds=%{customdata[1]}<br>"
                "seeds=%{customdata[2]}<extra></extra>"
            ),
        ))
    fig.update_layout(
        title=f"{title}<br><sup>dots = individual seeds; diamonds = mean over seeds; no uncertainty bars</sup>",
        template="plotly_white",
        width=1050,
        height=680,
        margin={"l": 80, "r": 220, "t": 105, "b": 70},
        legend={"orientation": "v", "yanchor": "top", "y": 1, "xanchor": "left", "x": 1.01},
    )
    fig.update_xaxes(title_text="avg action-0 fraction")
    fig.update_yaxes(title_text="test episodic survival (%)")
    save_figure(fig, save_name or title)
    if SHOW_FIGURES:
        fig.show()
    return fig, mean_rows.sort_values("mean_survival_percent", ascending=False)


def plot_all_runs_action0_survival_scatter(action_rows, survival_rows, run_groups, *, title, save_name=None, aggregate_seeds=True):
    run_to_label = {
        str(run_name): str(label)
        for label, runs in run_groups.items()
        for run_name in runs
    }
    selected_runs = set(run_to_label)
    action = action_rows[action_rows["run_name"].astype(str).isin(selected_runs)].copy()
    survival = survival_rows[survival_rows["run_name"].astype(str).isin(selected_runs)].copy()
    if action.empty or survival.empty:
        print(f"Missing action or survival rows for {title}")
        return None, pd.DataFrame()
    if "selected_metric_step" not in action.columns:
        action["selected_metric_step"] = np.nan
    if "download_group" not in action.columns:
        action["download_group"] = ""
    action["condition_label"] = action["run_name"].astype(str).map(run_to_label)
    survival["condition_label"] = survival["run_name"].astype(str).map(run_to_label)
    action_all = (
        action.groupby(["run_name", "family", "condition_label", "seed", "download_group"], as_index=False, observed=True)
        .agg(
            action0_all_agents=("action0_fraction", "mean"),
            min_action0_agent=("action0_fraction", "min"),
            max_action0_agent=("action0_fraction", "max"),
            n_agents=("agent", "nunique"),
            agents=("agent", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
            selected_metric_steps=("selected_metric_step", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
        )
    )
    survival_cols = ["run_name", "family", "condition_label", "seed", "survival_percent", "survival_metric", "selected_survival_step"]
    survival_cols = [col for col in survival_cols if col in survival.columns]
    merged = action_all.merge(
        survival[survival_cols],
        on=["run_name", "family", "condition_label", "seed"],
        how="inner",
    )
    if merged.empty:
        print(f"No merged action/survival rows for {title}")
        return None, merged
    label_order = [label for label in run_groups.keys() if label in set(merged["condition_label"])]
    label_order += [label for label in _order_labels(merged["condition_label"].unique()) if label not in label_order]
    palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24
    colors = {label: WCCI_COLOR.get(label, palette[idx % len(palette)]) for idx, label in enumerate(label_order)}
    seed_marker_symbols = {0: "circle", 1: "diamond", 2: "square", 3: "cross", 4: "x"}

    fig = go.Figure()
    if aggregate_seeds:
        def _flat_unique(values):
            out = []
            for value in values:
                candidates = value if isinstance(value, (list, tuple, set)) else [value]
                for item in candidates:
                    if pd.isna(item):
                        continue
                    if item not in out:
                        out.append(item)
            return sorted(out)

        plot_rows = (
            merged.groupby(["family", "condition_label"], as_index=False, observed=True)
            .agg(
                mean_action0_all_agents=("action0_all_agents", "mean"),
                std_action0_all_agents=("action0_all_agents", "std"),
                min_action0_all_agents=("action0_all_agents", "min"),
                max_action0_all_agents=("action0_all_agents", "max"),
                mean_survival_percent=("survival_percent", "mean"),
                std_survival_percent=("survival_percent", "std"),
                min_survival_percent=("survival_percent", "min"),
                max_survival_percent=("survival_percent", "max"),
                n_seeds=("seed", "nunique"),
                seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
                runs=("run_name", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
                download_groups=("download_group", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
                agents=("agents", _flat_unique),
                selected_metric_steps=("selected_metric_steps", _flat_unique),
                survival_metrics=("survival_metric", lambda values: sorted(pd.Series(values).dropna().astype(str).unique().tolist())),
                selected_survival_steps=("selected_survival_step", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
            )
        )
        plot_rows[["std_action0_all_agents", "std_survival_percent"]] = plot_rows[["std_action0_all_agents", "std_survival_percent"]].fillna(0.0)
        for label in label_order:
            label_rows = plot_rows[plot_rows["condition_label"].astype(str).eq(str(label))]
            if label_rows.empty:
                continue
            row = label_rows.iloc[0]
            fig.add_trace(go.Scatter(
                x=[row["mean_action0_all_agents"]],
                y=[row["mean_survival_percent"]],
                mode="markers+text",
                name=label,
                text=[label],
                textposition="top center",
                marker={"color": colors[label], "size": 13, "symbol": "diamond", "line": {"color": "black", "width": 1.1}, "opacity": 0.95},
                customdata=np.array([[
                    int(row["n_seeds"]),
                    str(row["seeds"]),
                    str(row["runs"]),
                    str(row["download_groups"]),
                    str(row["agents"]),
                    row["std_action0_all_agents"],
                    row["min_action0_all_agents"],
                    row["max_action0_all_agents"],
                    row["std_survival_percent"],
                    row["min_survival_percent"],
                    row["max_survival_percent"],
                    str(row["selected_metric_steps"]),
                    str(row["survival_metrics"]),
                    str(row["selected_survival_steps"]),
                ]], dtype=object),
                hovertemplate=(
                    f"<b>{label}</b><br>"
                    "mean action-0=%{x:.4f}<br>"
                    "std action-0=%{customdata[5]:.4f}<br>"
                    "min/max action-0=%{customdata[6]:.4f} / %{customdata[7]:.4f}<br>"
                    "mean survival=%{y:.2f}%<br>"
                    "std survival=%{customdata[8]:.2f}%<br>"
                    "min/max survival=%{customdata[9]:.2f}% / %{customdata[10]:.2f}%<br>"
                    "n_seeds=%{customdata[0]}<br>"
                    "seeds=%{customdata[1]}<br>"
                    "runs=%{customdata[2]}<br>"
                    "download_groups=%{customdata[3]}<br>"
                    "agents=%{customdata[4]}<br>"
                    "action metric steps=%{customdata[11]}<br>"
                    "survival metrics=%{customdata[12]}<br>"
                    "survival steps=%{customdata[13]}<extra></extra>"
                ),
            ))
        subtitle = "one point per run variant; seeds aggregated; no uncertainty bars"
        output_rows = plot_rows.sort_values("mean_survival_percent", ascending=False)
    else:
        for label in label_order:
            points = merged[merged["condition_label"].astype(str).eq(str(label))].sort_values(["seed", "run_name"])
            if points.empty:
                continue
            fig.add_trace(go.Scatter(
                x=points["action0_all_agents"],
                y=points["survival_percent"],
                mode="markers",
                name=label,
                marker={
                    "color": colors[label],
                    "size": 10,
                    "symbol": [seed_marker_symbols.get(int(seed), "circle") if pd.notna(seed) else "circle" for seed in points["seed"]],
                    "line": {"color": "white", "width": 1.1},
                    "opacity": 0.88,
                },
                customdata=points[[
                    "run_name",
                    "seed",
                    "download_group",
                    "n_agents",
                    "agents",
                    "min_action0_agent",
                    "max_action0_agent",
                    "survival_metric",
                    "selected_survival_step",
                    "selected_metric_steps",
                ]].to_numpy(dtype=object),
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    f"run type={label}<br>"
                    "seed=%{customdata[1]}<br>"
                    "download_group=%{customdata[2]}<br>"
                    "avg action-0=%{x:.4f}<br>"
                    "min/max agent action-0=%{customdata[5]:.4f} / %{customdata[6]:.4f}<br>"
                    "survival=%{y:.2f}%<br>"
                    "n_agents=%{customdata[3]}<br>"
                    "agents=%{customdata[4]}<br>"
                    "action metric steps=%{customdata[9]}<br>"
                    "survival metric=%{customdata[7]}<br>"
                    "survival step=%{customdata[8]}<extra></extra>"
                ),
            ))
        subtitle = "one point per seed run; x = mean action-0 over agents"
        output_rows = merged.sort_values(["condition_label", "seed", "run_name"], kind="stable")
    fig.update_layout(
        title=f"{title}<br><sup>{subtitle}</sup>",
        template="plotly_white",
        width=1200,
        height=760,
        hovermode="closest",
        legend={"orientation": "v", "yanchor": "top", "y": 1, "xanchor": "left", "x": 1.01},
        margin={"l": 80, "r": 300, "t": 105, "b": 80},
    )
    fig.update_xaxes(title_text="average action-0 fraction over agents", range=[0, 1.02])
    fig.update_yaxes(title_text="test episodic survival (%)")
    save_figure(fig, save_name or title)
    if SHOW_FIGURES:
        fig.show()
    return fig, output_rows


def load_full_test_action_summaries():
    root = TASK_DIR / "outputs" / "full_test_eval_actions"
    rows = []
    if not root.exists():
        return pd.DataFrame()
    for path in sorted(root.glob("*/action_summary.json")):
        run_like = path.parent.name
        if "wcci" not in run_like.lower():
            continue
        try:
            payload = json.loads(path.read_text())
        except Exception as exc:
            print(f"Skipped {path}: {exc}")
            continue
        stem = re.sub(r"_step\d+_job\d+$", "", run_like)
        stem = stem.removeprefix("best_test_")
        family = family_from_run(stem)
        seed = seed_from_run(stem)
        for agent, agent_data in payload.get("agents", {}).items():
            rows.append({
                "run_like": run_like,
                "run_name": stem,
                "family": family,
                "condition_label": pretty_label(family),
                "seed": seed,
                "agent": agent,
                "action0_fraction": agent_data.get("action0_fraction"),
                "global_step": payload.get("global_step"),
                "path": str(path),
            })
    return pd.DataFrame(rows)


## Editable WCCI Run Groups

In [73]:
# Edit this cell to choose which WCCI runs appear in each action plot.
BASELINE_FOR_COMPARISONS = "wcci_mlp_baseline_72x576_lr20m_f010"
BASELINE_LABEL = "MLP baseline 72x576 lr20M f0.10"
ACTION_METRIC_SOURCE = "test"  # "test", "train_eval", or "train"

WCCI_BASELINE_RUNS = {
    "MLP baseline 72x576": seeded("wcci_mlp_baseline_72x576"),
    BASELINE_LABEL: seeded(BASELINE_FOR_COMPARISONS),
    "MLP baseline 512x3 lr20M f0.10": seeded("wcci_mlp_baseline_512x512x512_72x576_lr20m_f010"),
    "MLP baseline GNN features": seeded("wcci_baseline_gnn_features_72x576"),
}
WCCI_HVG_RUNS = {
    BASELINE_LABEL: seeded(BASELINE_FOR_COMPARISONS),
    "global rho heuristic 0.90": seeded("wcci_hvg_01_eval_rho090_72x576_lr20m_f010"),
    "local rho heuristic 0.90": seeded("wcci_hvg_04_eval_local_rho090_72x576_lr20m_f010"),
    "gate final-action MAP": seeded("wcci_hvg_02_gate_final_map_72x576_lr20m_f010"),
    "gate hierarchical greedy": seeded("wcci_hvg_03_gate_hierarchical_72x576_lr20m_f010"),
}
WCCI_SPARSE16_RUNS = {
    BASELINE_LABEL: seeded(BASELINE_FOR_COMPARISONS),
    "Sparse16 flat p0.010": seeded("wcci_sparse16_flat_p010_72x576_lr20m_f010"),
}
WCCI_GNN_RUNS = {
    BASELINE_LABEL: seeded(BASELINE_FOR_COMPARISONS),
    "GINE light non-shared": seeded("wcci_gine_light_nonshared_72x576"),
    "MLP baseline GNN features": seeded("wcci_baseline_gnn_features_72x576"),
}
WCCI_AIB_RUNS = {
    BASELINE_LABEL: seeded(BASELINE_FOR_COMPARISONS),
    "AIB flat local t0.20 lr20m f0.10": seeded("wcci_aib_00_flat_local_t020_72x576_lr20m_f010"),
    "AIB flat local t0.10 lr20m f0.10": seeded("wcci_aib_01_flat_local_t010_72x576_lr20m_f010"),
}
ALL_COMPARISONS = {
    "WCCI baselines": WCCI_BASELINE_RUNS,
    "WCCI GNN": WCCI_GNN_RUNS,
    "WCCI HVG": WCCI_HVG_RUNS,
    "WCCI Sparse16": WCCI_SPARSE16_RUNS,
    "WCCI AIB": WCCI_AIB_RUNS,
}
ALL_WCCI_RUN_GROUPS = {}
for comparison_run_groups in ALL_COMPARISONS.values():
    for label, runs in comparison_run_groups.items():
        ALL_WCCI_RUN_GROUPS.setdefault(label, list(runs))

# Always filter by the downloaded source folder as well as run name.
COMPARISON_SOURCE_GROUPS = {
    "WCCI baselines": ["wcci_baseline_2"],
    "WCCI GNN": ["wcci_baseline_2", "wcci_gnn"],
    "WCCI HVG": ["wcci_baseline_2", "wcci_hvg_2"],
    "WCCI Sparse16": ["wcci_baseline_2", "wcci_sparse16_2"],
    "WCCI AIB": ["wcci_baseline_2", "wcci_aib_2"],
}


def source_history(comparison_name):
    sources = COMPARISON_SOURCE_GROUPS.get(comparison_name, WCCI_GROUPS)
    return history_df[history_df["download_group"].isin(sources)].copy()


def source_rows(rows, comparison_name):
    if rows is None or rows.empty or "download_group" not in rows.columns:
        return rows
    sources = COMPARISON_SOURCE_GROUPS.get(comparison_name, WCCI_GROUPS)
    return rows[rows["download_group"].isin(sources)].copy()


for name, groups in ALL_COMPARISONS.items():
    print(f"\n{name}")
    report_missing_runs(groups, source_history(name))



WCCI baselines


,curve,expected,available,missing
0,MLP baseline 72x576,3,3,[]
1,MLP baseline 72x576 lr20M f0.10,3,3,[]
2,MLP baseline 512x3 lr20M f0.10,3,3,[]
3,MLP baseline GNN features,3,3,[]



WCCI GNN


,curve,expected,available,missing
0,MLP baseline 72x576 lr20M f0.10,3,3,[]
1,GINE light non-shared,3,3,[]
2,MLP baseline GNN features,3,3,[]



WCCI HVG


,curve,expected,available,missing
0,MLP baseline 72x576 lr20M f0.10,3,3,[]
1,global rho heuristic 0.90,3,3,[]
2,local rho heuristic 0.90,3,3,[]
3,gate final-action MAP,3,3,[]
4,gate hierarchical greedy,3,3,[]



WCCI Sparse16


,curve,expected,available,missing
0,MLP baseline 72x576 lr20M f0.10,3,3,[]
1,Sparse16 flat p0.010,3,3,[]



WCCI AIB


,curve,expected,available,missing
0,MLP baseline 72x576 lr20M f0.10,3,3,[]
1,AIB flat local t0.20 lr20m f0.10,3,3,[]
2,AIB flat local t0.10 lr20m f0.10,3,3,[]


## Collect Latest Metrics

In [74]:
action0_latest = collect_latest_action0(source=ACTION_METRIC_SOURCE)
survival_latest = collect_latest_survival(split="test")
nonidle_latest = collect_nonidle_count_distribution()
full_test_action0 = load_full_test_action_summaries()

print(f"Latest action-0 rows from {ACTION_METRIC_SOURCE}: {len(action0_latest)}")
print(f"Latest survival rows: {len(survival_latest)}")
print(f"Latest non-idle-count rows: {len(nonidle_latest)}")
print(f"WCCI full-test action-summary rows: {len(full_test_action0)}")

if full_test_action0.empty:
    print("No WCCI full-test action summaries were found locally. The plots below use W&B history action metrics.")
else:
    display(full_test_action0.sort_values(["condition_label", "seed", "agent"]))


Latest action-0 rows from test: 144
Latest survival rows: 36
Latest non-idle-count rows: 180
WCCI full-test action-summary rows: 0
No WCCI full-test action summaries were found locally. The plots below use W&B history action metrics.


## WCCI Baselines

In [75]:
# Edit this cell to choose curves/runs for the plot below.
ACTION0_BASELINE_LABELS = [
    "MLP baseline 72x576",
    "MLP baseline 72x576 lr20M f0.10",
    "MLP baseline 512x3 lr20M f0.10",
    "MLP baseline GNN features",
]
ACTION0_BASELINE_RUN_NAMES = None  # e.g. ["wcci_mlp_baseline_72x576_lr20m_f010_s0"]
SELECTED_ACTION0_BASELINE_RUNS = select_plot_runs(
    WCCI_BASELINE_RUNS,
    labels=ACTION0_BASELINE_LABELS,
    run_names=ACTION0_BASELINE_RUN_NAMES,
)


In [76]:
fig_action0_baselines, action0_baselines_rows, action0_baselines_summary = plot_action0_by_agent(
    source_rows(action0_latest, "WCCI baselines"), SELECTED_ACTION0_BASELINE_RUNS,
    title=f"WCCI baselines: action-0 fraction by agent ({ACTION_METRIC_SOURCE})",
    save_name=f"wcci_baselines_action0_{ACTION_METRIC_SOURCE}",
)
# fig_action0_baselines

Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/wcci_baselines_action0_test.html


## WCCI Heuristic vs Gate

In [77]:
# Edit this cell to choose curves/runs for the plot below.
ACTION0_HVG_LABELS = [
    "MLP baseline 72x576 lr20M f0.10",
    "global rho heuristic 0.90",
    "local rho heuristic 0.90",
    "gate final-action MAP",
    "gate hierarchical greedy",
]
ACTION0_HVG_RUN_NAMES = None  # e.g. ["wcci_hvg_01_eval_rho090_72x576_lr20m_f010_s0"]
SELECTED_ACTION0_HVG_RUNS = select_plot_runs(
    WCCI_HVG_RUNS,
    labels=ACTION0_HVG_LABELS,
    run_names=ACTION0_HVG_RUN_NAMES,
)


In [78]:
fig_action0_hvg, action0_hvg_rows, action0_hvg_summary = plot_action0_by_agent(
    source_rows(action0_latest, "WCCI HVG"), SELECTED_ACTION0_HVG_RUNS,
    title=f"WCCI heuristic vs gate: action-0 fraction by agent ({ACTION_METRIC_SOURCE})",
    save_name=f"wcci_hvg_action0_{ACTION_METRIC_SOURCE}",
)
# fig_action0_hvg

Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/wcci_hvg_action0_test.html


## WCCI Sparse16

In [79]:
# Edit this cell to choose curves/runs for the plot below.
ACTION0_SPARSE16_LABELS = [
    "MLP baseline 72x576 lr20M f0.10",
    "Sparse16 flat p0.010",
]
ACTION0_SPARSE16_RUN_NAMES = None  # e.g. ["wcci_sparse16_flat_p010_72x576_lr20m_f010_s0"]
SELECTED_ACTION0_SPARSE16_RUNS = select_plot_runs(
    WCCI_SPARSE16_RUNS,
    labels=ACTION0_SPARSE16_LABELS,
    run_names=ACTION0_SPARSE16_RUN_NAMES,
)


In [80]:
fig_action0_sparse16, action0_sparse16_rows, action0_sparse16_summary = plot_action0_by_agent(
    source_rows(action0_latest, "WCCI Sparse16"), SELECTED_ACTION0_SPARSE16_RUNS,
    title=f"WCCI Sparse16: action-0 fraction by agent ({ACTION_METRIC_SOURCE})",
    save_name=f"wcci_sparse16_action0_{ACTION_METRIC_SOURCE}",
)
# fig_action0_sparse16

Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/wcci_sparse16_action0_test.html


## WCCI GNN

In [81]:
# Edit this cell to choose curves/runs for the plot below.
ACTION0_GNN_LABELS = [
    "MLP baseline 72x576 lr20M f0.10",
    "MLP baseline GNN features",
    "GINE light non-shared",
]
ACTION0_GNN_RUN_NAMES = None  # e.g. ["wcci_gine_light_nonshared_72x576_s0"]
SELECTED_ACTION0_GNN_RUNS = select_plot_runs(
    WCCI_GNN_RUNS,
    labels=ACTION0_GNN_LABELS,
    run_names=ACTION0_GNN_RUN_NAMES,
)


In [82]:
fig_action0_gnn, action0_gnn_rows, action0_gnn_summary = plot_action0_by_agent(
    source_rows(action0_latest, "WCCI GNN"), SELECTED_ACTION0_GNN_RUNS,
    title=f"WCCI GNN: action-0 fraction by agent ({ACTION_METRIC_SOURCE})",
    save_name=f"wcci_gnn_action0_{ACTION_METRIC_SOURCE}",
)
# fig_action0_gnn

Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/wcci_gnn_action0_test.html


## WCCI AIB

In [83]:
# Edit this cell to choose curves/runs for the plot below.
ACTION0_AIB_LABELS = [
    "MLP baseline 72x576 lr20M f0.10",
    "AIB flat local t0.20 lr20m f0.10",
    "AIB flat local t0.10 lr20m f0.10",
]
ACTION0_AIB_RUN_NAMES = None  # e.g. ["wcci_aib_00_flat_local_t020_72x576_lr20m_f010_s0"]
SELECTED_ACTION0_AIB_RUNS = select_plot_runs(
    WCCI_AIB_RUNS,
    labels=ACTION0_AIB_LABELS,
    run_names=ACTION0_AIB_RUN_NAMES,
)


In [84]:
fig_action0_aib, action0_aib_rows, action0_aib_summary = plot_action0_by_agent(
    source_rows(action0_latest, "WCCI AIB"), SELECTED_ACTION0_AIB_RUNS,
    title=f"WCCI AIB: action-0 fraction by agent ({ACTION_METRIC_SOURCE})",
    save_name=f"wcci_aib_action0_{ACTION_METRIC_SOURCE}",
)
# fig_action0_aib

Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/wcci_aib_action0_test.html


## Non-Idle Agent Count Distribution

In [85]:
# Edit this cell to choose curves/runs for the plot below.
NONIDLE_HVG_LABELS = [
    "MLP baseline 72x576 lr20M f0.10",
    "global rho heuristic 0.90",
    "local rho heuristic 0.90",
    "gate final-action MAP",
    "gate hierarchical greedy",
]
NONIDLE_HVG_RUN_NAMES = None  # e.g. ["wcci_hvg_01_eval_rho090_72x576_lr20m_f010_s0"]
SELECTED_NONIDLE_HVG_RUNS = select_plot_runs(
    WCCI_HVG_RUNS,
    labels=NONIDLE_HVG_LABELS,
    run_names=NONIDLE_HVG_RUN_NAMES,
)


In [86]:
fig_nonidle_hvg, nonidle_hvg_rows, nonidle_hvg_summary = plot_nonidle_distribution(
    source_rows(nonidle_latest, "WCCI HVG"), SELECTED_NONIDLE_HVG_RUNS,
    title="WCCI HVG: train non-idle-agent count distribution",
    save_name="wcci_hvg_nonidle_count_distribution",
)
# fig_nonidle_hvg

Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/wcci_hvg_nonidle_count_distribution.html


## Action-0 / Survival Tradeoffs

In [87]:
# Edit this cell to choose comparisons/curves for the tradeoff plots below.
TRADEOFF_COMPARISON_LABELS = [
    "WCCI baselines",
    "WCCI GNN",
    "WCCI HVG",
    "WCCI Sparse16",
    "WCCI AIB",
]
TRADEOFF_LABELS_BY_COMPARISON = {
    "WCCI baselines": list(WCCI_BASELINE_RUNS),
    "WCCI GNN": list(WCCI_GNN_RUNS),
    "WCCI HVG": list(WCCI_HVG_RUNS),
    "WCCI Sparse16": list(WCCI_SPARSE16_RUNS),
    "WCCI AIB": list(WCCI_AIB_RUNS),
}
TRADEOFF_RUN_NAMES_BY_COMPARISON = {
    "WCCI baselines": None,
    "WCCI GNN": None,
    "WCCI HVG": None,
    "WCCI Sparse16": None,
    "WCCI AIB": None,
}
SELECTED_TRADEOFF_RUNS = {
    comparison_name: select_plot_runs(
        ALL_COMPARISONS[comparison_name],
        labels=TRADEOFF_LABELS_BY_COMPARISON.get(comparison_name),
        run_names=TRADEOFF_RUN_NAMES_BY_COMPARISON.get(comparison_name),
    )
    for comparison_name in TRADEOFF_COMPARISON_LABELS
}


In [88]:
tradeoff_outputs = {}
for comparison_name, groups in SELECTED_TRADEOFF_RUNS.items():
    fig, rows = plot_action0_survival_tradeoff(
        source_rows(action0_latest, comparison_name), source_rows(survival_latest, comparison_name), groups,
        title=f"{comparison_name}: action-0 / survival tradeoff",
        save_name=f"{comparison_name}_action0_survival_tradeoff",
    )
    tradeoff_outputs[comparison_name] = rows.assign(comparison=comparison_name) if not rows.empty else rows

nonempty_tradeoffs = [df for df in tradeoff_outputs.values() if isinstance(df, pd.DataFrame) and not df.empty]
combined_tradeoff = pd.concat(nonempty_tradeoffs, ignore_index=True) if nonempty_tradeoffs else pd.DataFrame()
display(combined_tradeoff)

Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/WCCI_baselines_action0_survival_tradeoff.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/WCCI_GNN_action0_survival_tradeoff.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/WCCI_HVG_action0_survival_tradeoff.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/WCCI_Sparse16_action0_survival_tradeoff.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/WCCI_AIB_action0_survival_tradeoff.html


,family,condition_label,mean_action0_all_agents,std_action0_all_agents,mean_survival_percent,std_survival_percent,n_seeds,seeds,comparison
0,wcci_mlp_baseline_72x576,MLP baseline 72x576,0.136257,0.061603,27.877698,11.828475,3,"[0, 1, 2]",WCCI baselines
1,wcci_baseline_gnn_features_72x576,MLP baseline GNN features,0.142210,0.103769,25.871165,9.868138,3,"[0, 1, 2]",WCCI baselines
2,wcci_mlp_baseline_72x576_lr20m_f010,MLP baseline 72x576 lr20M f0.10,0.167709,0.048164,23.751757,7.702260,3,"[0, 1, 2]",WCCI baselines
3,wcci_mlp_baseline_512x512x512_72x576_lr20m_f010,MLP baseline 512x3 lr20M f0.10,0.099296,0.028410,20.026048,10.519005,3,"[0, 1, 2]",WCCI baselines
4,wcci_baseline_gnn_features_72x576,MLP baseline GNN features,0.142210,0.103769,25.871165,9.868138,3,"[0, 1, 2]",WCCI GNN
5,wcci_mlp_baseline_72x576_lr20m_f010,MLP baseline 72x576 lr20M f0.10,0.167709,0.048164,23.751757,7.702260,3,"[0, 1, 2]",WCCI GNN
6,wcci_gine_light_nonshared_72x576,GINE light non-shared,0.271575,0.110219,12.779294,2.424278,3,"[0, 1, 2]",WCCI GNN
7,wcci_hvg_01_eval_rho090_72x576_lr20m_f010,global rho heuristic 0.90,0.973661,0.023058,28.353180,11.060336,3,"[0, 1, 2]",WCCI HVG
8,wcci_hvg_02_gate_final_map_72x576_lr20m_f010,gate final-action MAP,0.400317,0.069300,25.922434,10.688363,3,"[0, 1, 2]",WCCI HVG
9,wcci_mlp_baseline_72x576_lr20m_f010,MLP baseline 72x576 lr20M f0.10,0.167709,0.048164,23.751757,7.702260,3,"[0, 1, 2]",WCCI HVG


## All WCCI Runs Scatter

In [89]:
# Edit this cell to choose which run variants/runs appear in the combined scatter below.
ALL_RUN_SCATTER_LABELS = list(ALL_WCCI_RUN_GROUPS)
ALL_RUN_SCATTER_RUN_NAMES = None  # e.g. ["wcci_mlp_baseline_72x576_lr20m_f010_s0"]
ALL_RUN_SCATTER_AGGREGATE_SEEDS = True
SELECTED_ALL_RUN_SCATTER_RUNS = select_plot_runs(
    ALL_WCCI_RUN_GROUPS,
    labels=ALL_RUN_SCATTER_LABELS,
    run_names=ALL_RUN_SCATTER_RUN_NAMES,
)


In [90]:
fig_all_wcci_run_scatter, all_wcci_run_scatter_rows = plot_all_runs_action0_survival_scatter(
    action0_latest,
    survival_latest,
    SELECTED_ALL_RUN_SCATTER_RUNS,
    title=f"All WCCI runs: action-0 / survival scatter ({ACTION_METRIC_SOURCE})",
    save_name=f"wcci_all_runs_action0_survival_scatter_{ACTION_METRIC_SOURCE}",
    aggregate_seeds=ALL_RUN_SCATTER_AGGREGATE_SEEDS,
)
if not all_wcci_run_scatter_rows.empty:
    display(all_wcci_run_scatter_rows)
fig_all_wcci_run_scatter

Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/wcci_all_runs_action0_survival_scatter_test.html


,family,condition_label,mean_action0_all_agents,std_action0_all_agents,min_action0_all_agents,max_action0_all_agents,mean_survival_percent,std_survival_percent,min_survival_percent,max_survival_percent,n_seeds,seeds,runs,download_groups,agents,selected_metric_steps,survival_metrics,selected_survival_steps
4,wcci_hvg_01_eval_rho090_72x576_lr20m_f010,global rho heuristic 0.90,0.973661,0.023058,0.947079,0.988270,28.353180,11.060336,17.068966,39.175143,3,"[0, 1, 2]","[wcci_hvg_01_eval_rho090_72x576_lr20m_f010_s0,...",[wcci_hvg_2],"[agent_0, agent_1, agent_2, agent_3]",[59968512],[test/episodic_survival],[59968512]
9,wcci_mlp_baseline_72x576,MLP baseline 72x576,0.136257,0.061603,0.091477,0.206512,27.877698,11.828475,14.810221,37.852890,3,"[0, 1, 2]","[wcci_mlp_baseline_72x576_s0, wcci_mlp_baselin...",[wcci_baseline_2],"[agent_0, agent_1, agent_2, agent_3]",[59968512],[test/episodic_survival],[59968512]
11,wcci_sparse16_flat_p010_72x576_lr20m_f010,Sparse16 flat p0.010,0.489060,0.024885,0.469942,0.517197,26.996196,12.128658,13.298189,36.370628,3,"[0, 1, 2]","[wcci_sparse16_flat_p010_72x576_lr20m_f010_s0,...",[wcci_sparse16_2],"[agent_0, agent_1, agent_2, agent_3]",[59968512],[test/episodic_survival],[59968512]
5,wcci_hvg_02_gate_final_map_72x576_lr20m_f010,gate final-action MAP,0.400317,0.069300,0.321617,0.452202,25.922434,10.688363,13.604565,32.747457,3,"[0, 1, 2]",[wcci_hvg_02_gate_final_map_72x576_lr20m_f010_...,[wcci_hvg_2],"[agent_0, agent_1, agent_2, agent_3]",[59968512],[test/episodic_survival],[59968512]
2,wcci_baseline_gnn_features_72x576,MLP baseline GNN features,0.142210,0.103769,0.076374,0.261830,25.871165,9.868138,14.584470,32.870256,3,"[0, 1, 2]","[wcci_baseline_gnn_features_72x576_s0, wcci_ba...",[wcci_baseline_2],"[agent_0, agent_1, agent_2, agent_3]",[59968512],[test/episodic_survival],[59968512]
10,wcci_mlp_baseline_72x576_lr20m_f010,MLP baseline 72x576 lr20M f0.10,0.167709,0.048164,0.128337,0.221411,23.751757,7.702260,14.961548,29.319028,3,"[0, 1, 2]","[wcci_mlp_baseline_72x576_lr20m_f010_s0, wcci_...",[wcci_baseline_2],"[agent_0, agent_1, agent_2, agent_3]",[59968512],[test/episodic_survival],[59968512]
0,wcci_aib_00_flat_local_t020_72x576_lr20m_f010,AIB flat local t0.20 lr20m f0.10,0.926510,0.019190,0.904768,0.941085,21.645167,2.213436,19.548499,23.959315,3,"[0, 1, 2]",[wcci_aib_00_flat_local_t020_72x576_lr20m_f010...,[wcci_aib_2],"[agent_0, agent_1, agent_2, agent_3]",[59968512],[test/episodic_survival],[59968512]
1,wcci_aib_01_flat_local_t010_72x576_lr20m_f010,AIB flat local t0.10 lr20m f0.10,0.904544,0.034823,0.873955,0.942441,20.891011,6.782900,13.058794,24.810221,3,"[0, 1, 2]",[wcci_aib_01_flat_local_t010_72x576_lr20m_f010...,[wcci_aib_2],"[agent_0, agent_1, agent_2, agent_3]","[56153088, 56899584, 57314304]",[test/episodic_survival],"[56153088, 56899584, 57314304]"
8,wcci_mlp_baseline_512x512x512_72x576_lr20m_f010,MLP baseline 512x3 lr20M f0.10,0.099296,0.028410,0.076618,0.131164,20.026048,10.519005,9.080873,30.059539,3,"[0, 1, 2]",[wcci_mlp_baseline_512x512x512_72x576_lr20m_f0...,[wcci_baseline_2],"[agent_0, agent_1, agent_2, agent_3]",[59968512],[test/episodic_survival],[59968512]
6,wcci_hvg_03_gate_hierarchical_72x576_lr20m_f010,gate hierarchical greedy,0.094997,0.033840,0.060984,0.128660,17.776400,3.700289,13.682709,20.883156,3,"[0, 1, 2]",[wcci_hvg_03_gate_hierarchical_72x576_lr20m_f0...,[wcci_hvg_2],"[agent_0, agent_1, agent_2, agent_3]",[59968512],[test/episodic_survival],[59968512]


## Optional Full-Test Action Summaries

In [91]:
# Edit this cell to choose rows for the optional full-test action-summary plot below.
FULL_TEST_ACTION0_CONDITIONS = None  # e.g. ["MLP baseline 72x576 lr20M f0.10"]
FULL_TEST_ACTION0_RUN_NAMES = None  # e.g. ["wcci_mlp_baseline_72x576_lr20m_f010_s0"]

selected_full_test_action0 = full_test_action0.copy()
if FULL_TEST_ACTION0_CONDITIONS is not None and not selected_full_test_action0.empty:
    selected_full_test_action0 = selected_full_test_action0[
        selected_full_test_action0["condition_label"].isin(FULL_TEST_ACTION0_CONDITIONS)
    ].copy()
if FULL_TEST_ACTION0_RUN_NAMES is not None and not selected_full_test_action0.empty:
    run_col = "run_like" if "run_like" in selected_full_test_action0.columns else "run_name"
    selected_full_test_action0 = selected_full_test_action0[
        selected_full_test_action0[run_col].isin(FULL_TEST_ACTION0_RUN_NAMES)
    ].copy()


In [92]:
if selected_full_test_action0.empty:
    print("No WCCI full-test action summaries found. Re-run this cell after adding outputs/full_test_eval_actions/*wcci*/action_summary.json.")
else:
    selected_full_test_run_groups = {
        label: sorted(group["run_name"].dropna().astype(str).unique().tolist())
        for label, group in selected_full_test_action0.groupby("condition_label", sort=False)
    }
    fig_full_test, full_test_action0_rows, full_test_summary = plot_action0_by_agent(
        selected_full_test_action0,
        selected_full_test_run_groups,
        title="WCCI full-test action-0 fraction by agent",
        save_name="wcci_full_test_action0_by_agent",
    )
    if not full_test_summary.empty:
        display(full_test_summary.sort_values(["condition_label", "agent"]))
    fig_full_test

No WCCI full-test action summaries found. Re-run this cell after adding outputs/full_test_eval_actions/*wcci*/action_summary.json.
